In [1]:
# imports!
import numpy as np
import pyvista as pv
from desc.dipole import _Dipole, DipoleSet, import_dipoles
from desc.io import load
import matplotlib.pyplot as plt
import jax.numpy as jnp

from scipy.optimize import minimize

In [2]:
eq = load('input.muse-fixedb_output.h5')[-1]
fin = 'muse_dipoles_desc.csv'
coilset = load('tf_coils_desc.h5')
one_period = import_dipoles(2, True, fin)

75460


In [6]:
rhos = np.array([dip.rho for dip in one_period.dipoles])
g = DipoleSet.calc_g(one_period, eq)

In [21]:
from desc.grid import LinearGrid
from desc.integrals import compute_B_plasma
from desc.plotting import plot_dipoles, plot_coils2


def compute_average_normalized_field(
    field, coils, eq, pl, vacuum=False, chunk_size=None, B_plasma_chunk_size=None
):
    if B_plasma_chunk_size is None:
        B_plasma_chunk_size = chunk_size
    grid = LinearGrid(M=20, N=20, NFP=eq.NFP, endpoint=True)
    
    surf_coords = eq.surface.compute(['R', 'phi', 'Z'], grid=grid)

    # asked chatgpt to make the surf coords array
    surf_coords_array = np.column_stack([surf_coords['R'], surf_coords['phi'], surf_coords['Z']])
    n_surf = eq.surface.compute(['n_rho'], grid=grid)['n_rho']
    
    B_PM = field.compute_magnetic_field(surf_coords_array)
    
    B_TF = coils.compute_magnetic_field(surf_coords_array)
    
    B_total = B_PM + B_TF
    
    Bn = np.sum(B_total * n_surf, axis=1)
    
    # compute normalization
    normalizing_field_vec = B_total
    if not vacuum:
        # add plasma field to the normalizing field
        normalizing_field_vec += compute_B_plasma(
            eq, eval_grid=grid, chunk_size=B_plasma_chunk_size
        )
    
    normalizing_field = np.mean(np.linalg.norm(normalizing_field_vec, axis=1))

    Bn_error_sq = (Bn / normalizing_field) ** 2


    ### total objective: B_normal = [ (B_PM + B_TF) dot n_surf / norm]**2 <- make this zero

    # make plot
    data = eq.surface.compute(['X','Y','Z'], grid=grid)
    X = np.asarray(data['X']).reshape(41,41)
    Y = np.asarray(data['Y']).reshape(41,41)
    Z = np.asarray(data['Z']).reshape(41,41)

    pgrid = pv.StructuredGrid(X,Y,Z)

    B_PM_dot_n = np.sum(B_PM * n_surf, axis=1).reshape(41, 41)
    B_TF_dot_n = np.sum(B_TF * n_surf, axis=1).reshape(41, 41)
    #pgrid["bn_error"] = (B_PM_dot_n+B_TF_dot_n).flatten()
    pgrid["bn_error"] = Bn_error_sq
    

    surf = pgrid.extract_surface()
    pl.add_mesh(
        surf,
        scalars="bn_error",
        cmap="viridis"
    )
    

    pl.add_mesh(pgrid, scalars="bn_error", cmap="viridis")

    return np.mean(np.abs(Bn)) 

In [4]:
pl = pv.Plotter()

In [22]:
print(compute_average_normalized_field(one_period,coilset,eq,pl))

/Users/jessicali/DESC/desc/utils.py:572: UserWarning: Could not build fft interpolator, switching to dft which is slow.
Reason: Got False for eval_grid.can_fft2.
  warnings.warn(msg, err)


0.019664734766773696


In [ ]:
0.0024461241181782164

In [7]:
grid = LinearGrid(M=8, N=12, NFP=eq.NFP, sym=False, endpoint=True)
n_surf = eq.surface.compute(['n_rho'], grid=grid)['n_rho']
coord = g[1]

In [23]:
def Bnormerror(rho, n_surf, coords):
    for i in range(len(rho)):
        one_period[i].rho = rho[i]
    #g = DipoleSet.calc_g(one_period, eq)
    #Bn_PM = g[0] @ rho #shape (425,)
    #B_coils = coilset.compute_magnetic_field(g[1], basis="xyz") #shape (425,3); 
                            #g[1] is the xyz points where inductance matrix was calculated
    #Bn_coils = jnp.sum(B_coils * n_surf, axis=1)
    #Bn = Bn_PM + Bn_coils
    B_total = one_period.compute_magnetic_field(coords, basis="xyz") + coilset.compute_magnetic_field(coords, basis="xyz")
    Bn = jnp.sum(B_total * n_surf, axis=1)
    #normalizing_field = jnp.mean(jnp.abs(Bn))
    #Bn_error_sq = (np.sum(Bn)/normalizing_field) ** 2
    #return Bn_error_sq
    print(jnp.mean(jnp.abs(Bn)))
    return np.asarray(jnp.mean(jnp.abs(Bn)))

In [24]:
import jax
gradient = np.asarray(jax.grad(Bnormerror))
#bounds = jnp.array([(-1.0,1.0)]*len(rhos))
bounds = np.array([(-1.0,1.0)]*len(rhos))


In [ ]:
opt = minimize(Bnormerror, x0=np.zeros(75460), args=(n_surf, coord), tol = 1e-12, method="TNC", bounds=bounds, jac = gradient, options={'maxfun':1000,'maxiter': 1,'disp': True})

/var/folders/wp/cyh_sjr513s3fv1yvb1rklpc0000gn/T/ipykernel_30919/1923450954.py:3: OptimizeWarning: Unknown solver options: maxiter
  opt = minimize(Bnormerror, x0=optimized_rhos, args=(n_surf, coord), tol = 1e-12, method="TNC", bounds=bounds, jac = gradient, options={'maxfun':1,'maxiter': 1,'disp': True})


0.053109553895606995
0.053109553895607724
0.053109553895609146
0.053109553895610506
0.053109553895611776
0.053109553895612935
0.05310955389561396
0.053109553895614864
0.0531095538956156
0.05310955389561622
0.053109553895616675
0.053109553895617015
0.05310955389561722
0.053109553895617355
0.05310955389561737
0.05310955389561732
0.05310955389561719
0.053109553895617015
0.05310955389561679
0.05310955389561651
0.05310955389561612
0.053109553895615766
0.05310955389561539
0.05310955389561498
0.053109553895614545
0.053109553895614114
0.053109553895613594
0.05310955389561311
0.05310955389561289
0.05310955389561241
0.053109553895611186
0.053109553895611235
0.05310955389561292
0.05310955389561034
0.05310955389560538
0.05310955389561195
0.05310955389561748
0.053109553895601846
0.05310955389559707
0.053109553895608244
0.05310955389560964
0.05310955389561095
0.05310955389561218
0.05310955389561327
0.05310955389561421
0.05310955389561504
0.05310955389561571
0.05310955389561625
0.053109553895616654
0

Exception ignored in: <function _xla_gc_callback at 0x11859e020>
Traceback (most recent call last):
  File "/Users/jessicali/qi-pm/lib/python3.12/site-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


0.05310955389561474
0.053109553895615405


In [ ]:
print(opt)

In [ ]:
print(np.sum(opt.x==0))

In [ ]:
print(opt.x[1000])

In [ ]:
print(np.mean(opt.x-rhos))

In [ ]:
for i in range(len(opt.x)):
    one_period[i].rho = opt.x[i]

In [ ]:
print(one_period[1000].rho)

In [ ]:
pip install "pyvista[jupyter]"

In [ ]:
pv.set_jupyter_backend('trame')

In [ ]:
x = np.zeros(75460)
y = np.zeros(75460)
z = np.zeros(75460)
for i in range(75460):
    x[i] = one_period[i].x
    y[i] = one_period[i].y
    z[i] = one_period[i].z

In [ ]:


#data = np.genfromtxt(fin, delimiter=',', skip_header=1)
#x,y,z,phi,theta,m,rho,I = data.T

#idx_nonzero = np.argwhere((I != 0) & (rho != 0))[:, 0]
#points = np.transpose([x,y,z])[idx_nonzero]
points = np.transpose([x,y,z])
cloud = pv.PolyData(points)

# optionally attach a scalar for coloring
#cloud["values"] = some_scalar_array  # shape (10000,)
b1 = compute_average_normalized_field(one_period, coilset, eq, pl)
print(b1)
pl.add_points(
    cloud,
    scalars=res.x,
    point_size=8,
    render_points_as_spheres=True,
    cmap="viridis",
)
#plot_dipoles(one_period, pl)

pl.add_scalar_bar()
pl.camera.azimuth = 90
#pv.set_jupyter_backend(None)
pl.show()